# DCG Multiview Clustering - Training Notebook

This notebook supports training **Incomplete Multi-View Clustering (IMVC)** models using the Diffusion Contrastive Generation (DCG) framework.

## Features

✅ **Local or Colab execution** - Automatically detects environment  
✅ **Automatic dataset download** - From Google Drive  
✅ **Hyperparameter tuning** - Via Optuna  
✅ **Flexible training** - Custom datasets, missing rates, epochs  
✅ **Metrics logging** - Saves ACC, NMI, ARI per epoch  

## Quick Start Flow

1. **Setup:** Run cells in "GPU & Environment Setup" section
2. **Data:** Run cells in "Download Datasets" section  
3. **Train:** Choose one of:
   - **Option A:** Run Tuning section (Optuna) → then Final Training with tuned params
   - **Option B:** Skip tuning and run Final Training with defaults
4. **Results:** Check printed metrics (ACC, NMI, ARI)

## Environment

- **GPU:** T4 or better recommended
- **Colab:** Change Runtime → GPU before running
- **Local:** Ensure in `DCG (NEW)` directory with dependencies installed

## Important Notes

- Datasets download to `../datasets/` relative to this notebook
- Training checkpoints save to `./checkpoints/`
- Metrics logged to CSV for post-hoc analysis
- Each trial/run is reproducible via seed parameter


---

## Updates in This Version (April 2026)

- ✅ Now uses **current DCG (NEW)** framework API
- ✅ Removed GitHub download (local/pre-cloned support)
- ✅ Updated all imports and function signatures
- ✅ Improved dataset download with progress tracking
- ✅ Better error handling and verbose output
- ✅ Supports multi-view datasets (2+ views)

---

## GPU & Environment Setup


In [1]:
import torch
import subprocess

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'])

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:
# Install gdown if needed
%pip install -q gdown

---

## Data Loading & Setup

Setup paths and verify data directory structure.


In [ ]:
import os
import sys
from pathlib import Path

# For local development: add current directory to path
# For Colab: this points to the cloned repo
current_dir = os.getcwd()
DCG_DIR = current_dir if os.path.exists(os.path.join(current_dir, 'train.py')) else '/content/Diffusion-based-approaches/DCG (NEW)'

if DCG_DIR not in sys.path:
    sys.path.insert(0, DCG_DIR)

DATASETS_DIR = os.path.join(os.path.dirname(DCG_DIR), 'datasets')
os.makedirs(DATASETS_DIR, exist_ok=True)

print("DCG Directory:", DCG_DIR)
print("Datasets Directory:", DATASETS_DIR)
print("Files in DCG:", os.listdir(DCG_DIR)[:5], "...")


Project downloaded to: /content/Diffusion-based-approaches
DCG path: /content/Diffusion-based-approaches/DCG (NEW)


In [ ]:
# Download datasets from Google Drive using gdown
import gdown
import os

DATASETS_DIR_PATH = DATASETS_DIR
os.makedirs(DATASETS_DIR_PATH, exist_ok=True)

# Dataset files: (file_id, filename)
DATASET_FILES = {
    "Synthetic3d": ("1sCgWYojsHjLqgEkijiD5XfreCIYw3APP", "Synthetic3d.mat"),
    "CUB": ("1T1ndPETHyenoBGFhVAQZbJUL725XK9G0", "CUB.mat"),
    "HandWritten": ("17OpkAo5_TTIjmE4FfB00WPXMkZlVSpb9", "Handwritten.mat"),
    "LandUse-21": ("19J6kxTYAvRHuCBrA7suPgc0Oe--C38e2", "LandUse-21.mat"),
    "Scene-15": ("1ENVvuQPFNceR6zhS-c53Qg4qCoFi6sFQ", "Scene-15.mat"),
    "NoisyMNIST": ("1jLgApO9_RvoIb96nzIgiq2nv8T1WjCbi", "NoisyMNIST.mat"),
}

def download_dataset(file_id, filename, dest_dir=DATASETS_DIR_PATH):
    """Download a single dataset from Google Drive."""
    output_path = os.path.join(dest_dir, filename)
    
    # Skip if already exists
    if os.path.exists(output_path):
        print(f"✓ {filename} already exists")
        return
    
    print(f"⏳ Downloading {filename}...")
    try:
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, output_path, quiet=False)
        print(f"✓ {filename} downloaded successfully\n")
    except Exception as e:
        print(f"✗ Failed to download {filename}: {e}\n")

# Download selected datasets (comment out to skip)
for name, (file_id, filename) in list(DATASET_FILES.items())[:2]:  # Download first 2 by default
    download_dataset(file_id, filename)

print("\nAvailable datasets in", DATASETS_DIR_PATH)
for f in os.listdir(DATASETS_DIR_PATH):
    if f.endswith('.mat'):
        size_mb = os.path.getsize(os.path.join(DATASETS_DIR_PATH, f)) / (1024*1024)
        print(f"  ✓ {f} ({size_mb:.1f} MB)")


Downloading...
From (original): https://drive.google.com/uc?id=1sCgWYojsHjLqgEkijiD5XfreCIYw3APP
From (redirected): https://drive.google.com/uc?id=1sCgWYojsHjLqgEkijiD5XfreCIYw3APP&confirm=t&uuid=650c7d20-cfbb-4012-a0f9-322a1f461206
To: /content/Diffusion-based-approaches/datasets/Synthetic3d.mat
100%|██████████| 53.8k/53.8k [00:00<00:00, 2.82MB/s]


Synthetic3d ready at /content/Diffusion-based-approaches/datasets/Synthetic3d.mat



Downloading...
From (original): https://drive.google.com/uc?id=1ENVvuQPFNceR6zhS-c53Qg4qCoFi6sFQ
From (redirected): https://drive.google.com/uc?id=1ENVvuQPFNceR6zhS-c53Qg4qCoFi6sFQ&confirm=t&uuid=d9154c4d-16eb-4070-9fc5-72980a8ed46f
To: /content/Diffusion-based-approaches/datasets/Scene-15.mat
100%|██████████| 3.38M/3.38M [00:00<00:00, 21.3MB/s]


Scene-15 ready at /content/Diffusion-based-approaches/datasets/Scene-15.mat



Downloading...
From (original): https://drive.google.com/uc?id=1jLgApO9_RvoIb96nzIgiq2nv8T1WjCbi
From (redirected): https://drive.google.com/uc?id=1jLgApO9_RvoIb96nzIgiq2nv8T1WjCbi&confirm=t&uuid=8101bda1-4b7f-4419-a32c-09d905a6bf35
To: /content/Diffusion-based-approaches/datasets/NoisyMNIST.mat
100%|██████████| 480M/480M [00:09<00:00, 48.1MB/s]


...

LandUse-21 ready at /content/Diffusion-based-approaches/datasets/LandUse-21.mat



Downloading...
From (original): https://drive.google.com/uc?id=17OpkAo5_TTIjmE4FfB00WPXMkZlVSpb9
From (redirected): https://drive.google.com/uc?id=17OpkAo5_TTIjmE4FfB00WPXMkZlVSpb9&confirm=t&uuid=d2d4d5fc-1887-4926-9919-c3911e471d9b
To: /content/Diffusion-based-approaches/datasets/Handwritten.mat
100%|██████████| 3.48M/3.48M [00:00<00:00, 25.8MB/s]


Handwritten ready at /content/Diffusion-based-approaches/datasets/Handwritten.mat



Downloading...
From (original): https://drive.google.com/uc?id=1T1ndPETHyenoBGFhVAQZbJUL725XK9G0
From (redirected): https://drive.google.com/uc?id=1T1ndPETHyenoBGFhVAQZbJUL725XK9G0&confirm=t&uuid=76f02526-5279-4247-ac3d-cf251246d7ba
To: /content/Diffusion-based-approaches/datasets/CUB.mat
100%|██████████| 3.18M/3.18M [00:00<00:00, 17.8MB/s]

CUB ready at /content/Diffusion-based-approaches/datasets/CUB.mat

All datasets ready in: /content/Diffusion-based-approaches/datasets


In [5]:
# Install required dependencies for this project
%pip install -q numpy scipy scikit-learn munkres

---

## Install Training Dependencies

Install required packages for training (PyTorch, scikit-learn, etc.)


In [ ]:
# Quick dataset presence check
import os

print('Data dir:', DATASETS_DIR)
required_datasets = ['Handwritten.mat', 'CUB.mat', 'Synthetic3d.mat']

found = []
for fname in os.listdir(DATASETS_DIR):
    if fname.endswith('.mat'):
        found.append(fname)

print('\nFound datasets:')
for f in sorted(found):
    path = os.path.join(DATASETS_DIR, f)
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f'  ✓ {f:20s} ({size_mb:6.1f} MB)')

if not found:
    print('  No datasets found. Please run the download cell above.')


Data dir: /content/Diffusion-based-approaches/datasets
Handwritten.mat -> FOUND
CUB.mat -> FOUND


---

## Verify Data Loading

Quick test to verify datasets are accessible and loader works correctly.


In [ ]:
# Check data loader with CLI test
import subprocess
import os

os.chdir(DCG_DIR)
result = subprocess.run([
    'python', 'check_loader_cli.py',
    '--dataset_root', DATASETS_DIR
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


/content/Diffusion-based-approaches/DCG (NEW)
main_dir= /content/Diffusion-based-approaches/DCG (NEW)
shuffle
OK cub: x=[(600, 1024), (600, 300)], y=[(600,)]
shuffle
OK landuse_21: x=[(2100, 20), (2100, 59), (2100, 40)], y=[(2100,)]
shuffle
OK handwritten: x=[(2000, 76), (2000, 240)], y=[(2000,)]
shuffle
OK synthetic3d: x=[(600, 3), (600, 3)], y=[(600,)]
shuffle
FAIL noisymnist: ValueError: Unsupported dataset name: noisymnist


## Hyperparameter Tuning with Optuna

Run this section to search for the best `lamda_mmi`, `lamda_diff`, and `lamda_cluster` values before your full training run.

**Notes:**
- Starts with `Synthetic3d` for quick iteration (fastest dataset)
- Increase `N_TRIALS` for stronger results
- Each trial runs only `TUNE_EPOCHS` (20 by default) for fast iteration
- The objective is the mean of `(ACC + NMI + ARI) / 3`
- Best parameters are saved to `best_lambda_params.json`


In [10]:
%pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.2 MB/s eta 0:00:00


In [ ]:
import os
import json
import torch
import optuna
import numpy as np
from sklearn.cluster import KMeans

os.chdir(DCG_DIR)
from datasets import load_data
from configure import get_default_config
from trainer import train, get_embeddings
from utils import build_models, build_optimizer, prepare_inputs, set_run_seed
from evaluation import get_cluster_sols, evaluation

# Tuning controls
SEED = 42
N_TRIALS = 10  # Reduced for faster iteration
TUNE_EPOCHS = 20
TUNE_MISSING_RATE = 0.2
TUNE_DATASET = 'Synthetic3d'  # Start with synthetic for speed

GPU_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device('cuda:0' if GPU_AVAILABLE else 'cpu')
print('Tuning device:', DEVICE)


def objective(trial):
    """Optimize lambda weights and return the mean clustering score."""
    config = get_default_config(TUNE_DATASET)
    config['dataset'] = TUNE_DATASET
    config['dataset_root'] = DATASETS_DIR
    config['training']['epoch'] = TUNE_EPOCHS
    config['training']['missing_rate'] = TUNE_MISSING_RATE

    config['training']['lamda_mmi'] = trial.suggest_float('lamda_mmi', 0.1, 5.0, log=True)
    config['training']['lamda_diff'] = trial.suggest_float('lamda_diff', 0.1, 5.0, log=True)
    config['training']['lamda_cluster'] = trial.suggest_float('lamda_cluster', 0.1, 5.0, log=True)

    run_seed = SEED + trial.number
    set_run_seed(run_seed, GPU_AVAILABLE)

    x_list, y_list = load_data(config)
    labels = y_list[0]
    x_views, mask = prepare_inputs(
        x_list,
        missing_rate=config['training']['missing_rate'],
        device=DEVICE,
        seed=config['training'].get('mask_seed'),
        min_view_presence_ratio=config['training'].get('min_view_presence_ratio', 0.1),
    )

    autoencoders, attention_layer, dfs, scheduler, imputer, view_head = build_models(config, DEVICE)
    optimizer = build_optimizer(autoencoders, attention_layer, dfs, view_head, lr=config['training']['lr'])

    try:
        train(
            autoencoders=autoencoders,
            attention_layer=attention_layer,
            dfs=dfs,
            scheduler=scheduler,
            imputer=imputer,
            view_head=view_head,
            x_views=x_views,
            mask=mask,
            optimizer=optimizer,
            device=DEVICE,
            n_epochs=TUNE_EPOCHS,
            batch_size=config['training']['batch_size'],
            n_clusters=config['training']['n_clusters'],
            lamda_recon=config['training'].get('lamda_recon', 1.0),
            lamda_mmi=config['training'].get('lamda_mmi', 1.0),
            lamda_diff=config['training'].get('lamda_diff', 1.0),
            lamda_cluster=config['training'].get('lamda_cluster', 1.0),
            mmi_temperature=1.0,
            conf_threshold=0.6,
            cluster_temperature=0.05,
            contrastive_temp=0.1,
            source_mode=config['training'].get('source_mode', 'attention'),
            return_history=False,
            verbose=False,
        )
    except Exception as e:
        print(f"Trial {trial.number} failed: {e}")
        return 0.0

    z_fused = get_embeddings(
        autoencoders=autoencoders,
        attention_layer=attention_layer,
        imputer=imputer,
        x_views=x_views,
        mask=mask,
        device=DEVICE,
        batch_size=512,
        source_mode=config['training'].get('source_mode', 'attention'),
    )

    z_np = z_fused.detach().cpu().numpy()
    labels_np = labels if isinstance(labels, np.ndarray) else np.array(labels)
    y_pred, _ = get_cluster_sols(z_np, ClusterClass=KMeans, n_clusters=config['training']['n_clusters'], init_args={'n_init': 20})
    scores = evaluation(y_pred, labels_np)

    acc, nmi, ari = scores['accuracy'], scores['NMI'], scores['ARI']
    trial.set_user_attr('acc', float(acc))
    trial.set_user_attr('nmi', float(nmi))
    trial.set_user_attr('ari', float(ari))

    mean_score = float((acc + nmi + ari) / 3.0)
    print(f"Trial {trial.number}: ACC={acc:.3f}, NMI={nmi:.3f}, ARI={ari:.3f}, Mean={mean_score:.3f}")
    return mean_score


print(f"\n{'='*60}")
print("Starting Optuna hyperparameter tuning")
print(f"Dataset: {TUNE_DATASET}, Missing rate: {TUNE_MISSING_RATE}")
print(f"Epochs per trial: {TUNE_EPOCHS}, Max trials: {N_TRIALS}")
print(f"{'='*60}\n")

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print('\n' + '='*60)
print('Tuning Complete!')
print('='*60)
print('Best score:', f"{study.best_value:.4f}")
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v:.4f}')

best_params_path = os.path.join(DCG_DIR, 'best_lambda_params.json')
with open(best_params_path, 'w', encoding='utf-8') as f:
    json.dump(study.best_params, f, indent=2)
print(f'\nSaved to: {best_params_path}')


[I 2026-04-02 23:34:31,183] A new study created in memory with name: no-name-61481dc5-50cc-45cd-a1ca-10a35dce58ab


Tuning device: cuda:0
shuffle


[W 2026-04-02 23:34:39,895] Trial 0 failed with parameters: {'lamda_mmi': 0.0014934929223038402, 'lamda_diff': 0.0002377213478518085, 'lamda_cluster': 0.00023237186372452727} because of the following error: RuntimeError('indices should be either on cpu or on the same device as the indexed tensor (cpu)').
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_35108/1398991412.py", line 61, in objective
    train_phase2(
  File "/content/Diffusion-based-approaches/DCG (NEW)/trainer.py", line 214, in train_phase2
    z_hat0 = scheduler.reconstruct_x0(z_noisy, t, noise_pred)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Diffusion-based-approaches/DCG (NEW)/baseModels.py", line 228, in reconstruct_x0
    s1 = self.sqrt_inv_alphas_bar[t].to(x_t.device)
         ~~~~~~~~~~~~~~~~~~~~~~~~^^^
Ru

RuntimeError: indices should be either on cpu or on the same device as the indexed tensor (cpu)

---

## Load Best Tuned Hyperparameters

After tuning completes, best parameters are automatically saved. The final training cell will load them if they exist. You can also modify them here manually.


In [ ]:
import os
import torch
from pathlib import Path
from argparse import Namespace

os.chdir(DCG_DIR)
from train import main

# Final Training Settings
TARGET_DATASET = 'Synthetic3d'
TARGET_MISSING_RATE = 0.3
FINAL_EPOCHS = 50

print(f"\n{'='*60}")
print(f"Starting Final Training with Tuned Hyperparameters")
print(f"{'='*60}")
print(f"Dataset: {TARGET_DATASET}")
print(f"Missing rate: {TARGET_MISSING_RATE}")
print(f"Final epochs: {FINAL_EPOCHS}")
print(f"Dataset root: {DATASETS_DIR}")
print(f"{'='*60}\n")

# Build args using tuned lambdas if available
tuned_lambdas = {}
best_params_path = os.path.join(DCG_DIR, 'best_lambda_params.json')
if os.path.exists(best_params_path):
    import json
    with open(best_params_path, 'r') as f:
        tuned_lambdas = json.load(f)
    print("Using tuned hyperparameters:")
    for k, v in tuned_lambdas.items():
        print(f"  {k}: {v:.4f}")
    print()

args = Namespace(
    dataset=TARGET_DATASET,
    dataset_root=DATASETS_DIR,
    missing_rate=TARGET_MISSING_RATE,
    epoch=FINAL_EPOCHS,
    seed=42,
    batch_size=64,
    lr=1e-3,
    mask_seed=123,
    lamda_recon=tuned_lambdas.get('lamda_recon', 1.0),
    lamda_mmi=tuned_lambdas.get('lamda_mmi', 1.0),
    lamda_diff=tuned_lambdas.get('lamda_diff', 1.0),
    lamda_cluster=tuned_lambdas.get('lamda_cluster', 1.0),
    save_dir=os.path.join(DCG_DIR, 'checkpoints'),
    input_dims=None,
    n_clusters=None,
    latent_dim=None,
    source_mode='attention'
)

# Run training
try:
    acc, nmi, ari = main(args)
    print(f"\n{'='*60}")
    print(f"Final Results:")
    print(f"  ACC: {acc*100:.2f}%")
    print(f"  NMI: {nmi*100:.2f}%")
    print(f"  ARI: {ari*100:.2f}%")
    print(f"{'='*60}")
except Exception as e:
    print(f"Training failed with error: {e}")
    import traceback
    traceback.print_exc()


Best tuned params loaded from: /content/Diffusion-based-approaches/DCG (NEW)/best_lambda_CUB_mr0p3.json
lambda_rec: 0.016763017695680008
lambda_df: 0.012019772763377443
lambda_ce: 0.011260786597807037
lambda_mmi: 0.004143066788712666
lambda_cluster: 0.7907078058336803
lambda_hc: 0.30394086062856396
mmi_temperature: 0.050753412249944105
mmi_internal_lambda: 1.9602313704350798


---

## Final Training (Option A: With Tuned Params)

Run this if you completed the tuning section. Uses best hyperparameters found during tuning.

**Dataset & settings:** Customize `TARGET_DATASET`, `TARGET_MISSING_RATE`, `FINAL_EPOCHS` below.


In [ ]:
import os
import torch
from pathlib import Path
from argparse import Namespace

os.chdir(DCG_DIR)
from train import main

# Alternative Final Training (without tuning - use defaults)
TARGET_DATASET = 'CUB'
TARGET_MISSING_RATE = 0.3
FINAL_EPOCHS = 100

print(f"\n{'='*60}")
print(f"Starting Training (using default hyperparameters)")
print(f"{'='*60}")
print(f"Dataset: {TARGET_DATASET}")
print(f"Missing rate: {TARGET_MISSING_RATE}")
print(f"Final epochs: {FINAL_EPOCHS}")
print(f"Dataset root: {DATASETS_DIR}")
print(f"{'='*60}\n")

args = Namespace(
    dataset=TARGET_DATASET,
    dataset_root=DATASETS_DIR,
    missing_rate=TARGET_MISSING_RATE,
    epoch=FINAL_EPOCHS,
    seed=42,
    batch_size=64,
    lr=1e-3,
    mask_seed=123,
    lamda_recon=1.0,
    lamda_mmi=1.0,
    lamda_diff=1.0,
    lamda_cluster=1.0,
    save_dir=os.path.join(DCG_DIR, 'checkpoints'),
    input_dims=None,
    n_clusters=None,
    latent_dim=None,
    source_mode='attention'
)

try:
    acc, nmi, ari = main(args)
    print(f"\n{'='*60}")
    print(f"Final Results:")
    print(f"  ACC: {acc*100:.2f}%")
    print(f"  NMI: {nmi*100:.2f}%")
    print(f"  ARI: {ari*100:.2f}%")
    print(f"{'='*60}")
except Exception as e:
    print(f"Training failed with error: {e}")
    import traceback
    traceback.print_exc()


/content/Diffusion-based-approaches/DCG (NEW)
GPU: True
Applied tuned params from best_lambda_CUB_mr0p3.json
  lambda_ce: 0.011260786597807037
  lambda_cluster: 0.7907078058336803
  lambda_df: 0.012019772763377443
  lambda_hc: 0.30394086062856396
  lambda_mmi: 0.004143066788712666
  lambda_rec: 0.016763017695680008
  mmi_internal_lambda: 1.9602313704350798
  mmi_temperature: 0.050753412249944105
Data set: CUB
Checkpoint dir: /content/Diffusion-based-approaches/DCG (NEW)/checkpoints
shuffle
--------------------Missing rate = 0.3--------------------
Resumed from checkpoint: checkpoints/CUB_mr0p3_seed1_last.pt
Epoch: 0/20 ==> loss = 1.0856 | rec=0.5580 df=0.1205 ce=5.4716 mmi=-3.9498 clu=1.2988 hc=0.0086
Epoch: 1/20 ==> loss = 1.0776 | rec=0.5663 df=0.1124 ce=5.5087 mmi=-3.9488 clu=1.2881 hc=0.0086
Epoch: 2/20 ==> loss = 1.0773 | rec=0.5574 df=0.1174 ce=5.4945 mmi=-3.9523 clu=1.2881 hc=0.0084
Epoch: 3/20 ==> loss = 1.1054 | rec=0.5552 df=0.1055 ce=5.5099 mmi=-3.9478 clu=1.3235 hc=0.0086
E

In [ ]:
# Plot training metrics from saved CSV files
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Find metrics files
checkpoint_dir = os.path.join(DCG_DIR, 'checkpoints')
if not os.path.exists(checkpoint_dir):
    print(f"Checkpoint directory not found: {checkpoint_dir}")
    print("Run a training cell first to generate metrics.")
else:
    # Look for CSV files with metrics
    csv_files = list(Path(checkpoint_dir).glob('*_metrics.csv'))
    
    if not csv_files:
        print(f"No metrics CSV files found in {checkpoint_dir}")
        print("Available files:", os.listdir(checkpoint_dir))
    else:
        print(f"Found {len(csv_files)} metrics file(s)")
        
        # Load and display first file
        df = pd.read_csv(csv_files[0])
        print(f"\nMetrics shape: {df.shape}")
        print("Columns:", df.columns.tolist())
        print("\nFirst few rows:")
        print(df.head())
        print("\nLast few rows:")
        print(df.tail())


Metrics directory: /content/Diffusion-based-approaches/DCG (NEW)/checkpoints
Found metric CSV files: 2
 - CUB_mr0p3_seed1_metrics.csv
 - CUB_mr0p3_seed2_metrics.csv


---

## Post-Training Analysis

Load and inspect training metrics saved during training. This shows epoch-by-epoch loss and clustering metrics.
